## 2.5,1 例子
根据设计 好的模型，系统会构建一个计算图(computational graph)，来跟踪计算是哪些数据通过哪些操作组合起来 产生输出。
一个标量函数关于向量x的梯度是向量，并且与x具有相同的形状。

In [1]:
import torch

x=torch.arange(4.0)

#手动开启梯度，训练时，只有模型参数（权重 w、偏置 b）​需要梯度
#PyTorch 会记录所有以 x 为参与者的运算，构建计算图，之后才能调用 backward() 反向传播出 x.grad。
#带后面下划线的是原地操作
x.requires_grad_(True) #等价于x=torch.arange(4.0,requires_grad=True) 需不需要计算梯度
#一个属性，存放梯度，默认值None，只用经过反向传播后才会被填入具体的值
x.grad 

y=2*torch.dot(x,x)
print(y)

#求的是 y 对 x 的梯度，偏导就是4*xi
#反向传播：梯度信号从输出 y 出发，沿计算图从后往前逐层回传，每层乘以本层导数，最后到达输入 x。
#y 是标量时不用传参：y.backward()。如果 y 是向量，得写 y.backward(v) 指定权重
y.backward()
#x.grad 会累加，所以每次反向传播前需要清零
print(x.grad)

tensor(28., grad_fn=<MulBackward0>)
tensor([ 0.,  4.,  8., 12.])


In [2]:
x.grad.zero_()
#各个分量加起来。求梯度就是求各个分量的偏导数
y=x.sum()
y.backward()
print(x.grad)

tensor([1., 1., 1., 1.])


## 2.5,2 非标量变量的反向传播
前面y是标量
> 损失函数（loss）的输出天然是标量

In [3]:
x.grad.zero_()
y=x*x
#PyTorch 的设计是：backward 只算“一个标量对 x 的梯度”
#必须先把向量 y 归约成一个标量，或者告诉它“怎么把这些输出合并”。
#数学上是对x的分量先平方再求和，所以对x的分量求偏导数都是2xi
y.sum().backward() 
# 等价于y.backward(torch.ones(len(x)))
#一个和 y 同形状的权重向量，作用是：把 y 的每个分量按权重加权求和，变成一个标量，再对这个标量求导。
print(x.grad)

tensor([0., 2., 4., 6.])


## 2.5.3 分离计算

In [ ]:
x.grad.zero_()
y=x*x
u=y.detach()
#detach() 方法返回一个新的张量，从当前计算图中分离出来，但是仍指向原变量的存放位置，不同之处只是requires_grad为false，得到的这个新变量，不再需要计算梯度，不具有grad。
#所以，detach()常用于不计算梯度的场合，以节省计算和存储空间。
#detach()返回的新变量与原变量共享内存，即修改一个，另一个也会跟着修改。
#返回一个和 y 数值完全相同的新张量，但它和 x 的计算关系被切断了
z=u*x #u被当成了一个常数 [0, 1, 4, 9]
z.sum().backward()
print(x.grad)

tensor([0., 1., 4., 9.])


In [5]:
x.grad.zero_()
y.sum().backward()
print(x.grad == 2 * x)



tensor([True, True, True, True])


## 2.5.4 Python控制流的梯度计算
即使构建函数的计算图需要通过Python控制流(例如，条件、循环或任意函数 调用)，我们仍然可以计算得到的变量的梯度。  
即使函数里带了循环和分支，自动求导照样能算梯度

In [ ]:
def f(a):
    b=a*2
    #不停地翻倍，直到 |b| ≥ 1000 为止。
    #norm求范数
    while b.norm()<1000:
        b=b*2
    if b.sum()>0:
        c=b
    else:
        c=100*b
    return c

#生成一个 0 到 1 之间的随机标量（形状是 () 的 0 维张量，这就是标量）
#size设定形状 (3,)1维向量，3个数
a=torch.rand(size=(),requires_grad=True)
print(a)
d=f(a)
d.backward()
a.grad 

tensor(0.2537, requires_grad=True)


tensor(4096.)